# 03. Характеристики удара и нелинейные модели

Что добавляют часть тела, техника, тип ситуации и готовые флаги StatsBomb.
И помогают ли деревья там, где линейная модель не справляется.

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

In [2]:
from xg_context.config import TABLES_DIR
from xg_context.visualization import apply_style

apply_style()
metrics = pd.read_csv(TABLES_DIR / 'model_metrics.csv')
metrics[['model', 'feature_set', 'validation_log_loss', 'test_log_loss', 'test_brier', 'test_roc_auc']].round(5)

,model,feature_set,validation_log_loss,test_log_loss,test_brier,test_roc_auc
0,statsbomb_xg (benchmark),—,0.24996,0.24042,0.06824,0.83458
1,Логистическая регрессия,geometry_shot_flags_defensive,0.25033,0.24271,0.06922,0.83004
2,Логистическая регрессия,geometry_shot_flags_defensive_no_visible,0.24998,0.24315,0.06941,0.82995
3,Случайный лес,geometry_shot_flags_defensive,0.25197,0.24325,0.06945,0.82964
4,Случайный лес,geometry_shot_flags_defensive_no_visible,0.25182,0.24366,0.06968,0.82954
5,Градиентный бустинг,geometry_shot_flags,0.25694,0.25234,0.07172,0.80827
6,Случайный лес,geometry_shot_flags,0.25676,0.25400,0.07223,0.80494
7,Логистическая регрессия,geometry_shot_flags,0.25515,0.25456,0.07209,0.80219
8,Логистическая регрессия,geometry_shot,0.25775,0.25630,0.07268,0.80068
9,Случайный лес,geometry_shot,0.26203,0.25783,0.07356,0.80174


## Выбор нелинейной модели

Выбор сделан по validation, тест в отборе не участвует.
Дерево, случайный лес и градиентный бустинг сравниваются между собой.
Набор признаков у всех один:
`geometry_shot_flags`.

In [3]:
nonlinear = metrics[
    metrics['feature_set'].eq('geometry_shot_flags')
]
nonlinear[['model', 'validation_log_loss', 'test_log_loss', 'best_params']]

,model,validation_log_loss,test_log_loss,best_params
5,Градиентный бустинг,0.256935,0.252336,"{""learning_rate"": 0.03, ""max_iter"": 200, ""max_..."
6,Случайный лес,0.256762,0.253998,"{""max_depth"": 14, ""min_samples_leaf"": 5, ""n_es..."
7,Логистическая регрессия,0.255152,0.254564,"{""C"": 10.0}"
10,Дерево решений,0.265713,0.259965,"{""max_depth"": 5, ""min_samples_leaf"": 50}"


Случайный лес и градиентный бустинг почти неразличимы по validation.
Выбран случайный лес.
Одиночное дерево заметно слабее.
Это ожидаемо: оно даёт кусочно-постоянные вероятности и хуже калибруется.

## Характеристики удара против готовых флагов

Признаки, которые я считаю сам, и разметку провайдера я держу отдельно.
Здесь видно, что даёт каждая группа.

In [4]:
ablation = pd.read_csv(TABLES_DIR / 'ablation.csv')
ablation[['уровень', 'описание', 'модель', 'test_log_loss', 'Δ_log_loss_шаг']].round(5)

,уровень,описание,модель,test_log_loss,Δ_log_loss_шаг
0,L1,геометрия удара,Логистическая регрессия,0.27282,NaN
1,L2,+ характеристики удара,Логистическая регрессия,0.25630,-0.01653
2,L3,+ флаги StatsBomb,Логистическая регрессия,0.25456,-0.00173
3,L4,+ свой защитный контекст,Логистическая регрессия,0.24271,-0.01185
4,L1,геометрия удара,Случайный лес,0.27206,NaN
5,L2,+ характеристики удара,Случайный лес,0.25783,-0.01423
6,L3,+ флаги StatsBomb,Случайный лес,0.25400,-0.00383
7,L4,+ свой защитный контекст,Случайный лес,0.24325,-0.01075


Переход L2 -> L3 добавляет `under_pressure`, `one_on_one` и `open_goal`.
Прирост минимальный, и bootstrap показывает, что он статистически неубедителен.
Значит, готовые флаги дали меньший прирост, чем мой защитный контекст.
Утверждать, что мои признаки не дублируют разметку StatsBomb, по этому сравнению нельзя.
Это два отдельных сравнения, а не проверка на дублирование.

In [5]:
pd.read_csv(TABLES_DIR / 'bootstrap.csv')[['сравнение', 'delta_log_loss', 'delta_log_loss_ci_low', 'delta_log_loss_ci_high', 'delta_log_loss_significant']].round(5)

,сравнение,delta_log_loss,delta_log_loss_ci_low,delta_log_loss_ci_high,delta_log_loss_significant
0,Логистическая: + характеристики удара (L1 → L2),-0.01653,-0.02085,-0.01186,True
1,Логистическая: + флаги StatsBomb (L2 → L3),-0.00173,-0.00350,0.00021,False
2,Логистическая: + защитный контекст (L3 → L4),-0.01185,-0.01603,-0.00784,True
3,Случайный лес: + защитный контекст (L3 → L4),-0.01075,-0.01532,-0.00629,True
4,Логистическая: + защитный контекст БЕЗ n_oppon...,-0.01141,-0.01553,-0.00736,True
5,Случайный лес: + защитный контекст БЕЗ n_oppon...,-0.01034,-0.01489,-0.00620,True
6,Вклад самого n_opponents_visible (L4− → L4),-0.00044,-0.00091,0.00002,False
7,Логистическая с защитным контекстом против sta...,0.00229,-0.00089,0.00522,False
